# Google Stock Price Time Series Analysis

This notebook performs time series forecasting on Google stock prices using three different approaches:
1. **LSTM (Long Short-Term Memory)** - Deep Learning approach
2. **ARIMA** - Statistical approach
3. **Prophet** - Facebook's time series forecasting tool

We'll compare the performance of these methods and make 30-day future predictions.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

Matplotlib is building the font cache; this may take a moment.


## 2. Load and Prepare Data

Load the Google stock data and prepare it for time series analysis.

In [2]:
# Load data from CSV
df = pd.read_csv('data/google.csv')

# Convert date column to datetime and sort
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

# Display basic information
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

Dataset shape: (5378, 6)
Date range: 2004-08-19 00:00:00 to 2026-01-02 00:00:00


,open,high,low,close,adj_close,volume
date,,,,,,
2004-08-19,2.492180,2.511011,2.604104,2.401401,2.502503,893181924
2004-08-20,2.690134,2.710460,2.729730,2.515015,2.527778,456686856
2004-08-23,2.717207,2.737738,2.839840,2.728979,2.771522,365122512
2004-08-24,2.604694,2.624374,2.792793,2.591842,2.783784,304946748
2004-08-25,2.632761,2.652653,2.702703,2.599600,2.626627,183772044


## 3. Method 1: LSTM (Deep Learning)

LSTM (Long Short-Term Memory) networks are a type of recurrent neural network capable of learning long-term dependencies, making them ideal for time series forecasting.

### 3.1 Import LSTM Libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

ModuleNotFoundError: No module named 'tensorflow'

### 3.2 Create Sequence Generator Function

This function creates sequences of data for training the LSTM model. We use past data points to predict the next value.

In [ ]:
def create_sequences(data, seq_length):
    """
    Create sequences for LSTM training.
    
    Parameters:
    - data: scaled time series data
    - seq_length: number of past time steps to use for prediction
    
    Returns:
    - X: input sequences
    - y: target values
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

### 3.3 Prepare Data for LSTM

Scale the data using MinMaxScaler and split into training and testing sets (80/20 split).

In [ ]:
# Prepare target variable (close price)
target = df['close'].values.reshape(-1, 1)

# Scale data to range [0, 1]
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(target)

# Split data (80% train, 20% test)
train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

print(f"Training samples: {len(train_data)}")
print(f"Testing samples: {len(test_data)}")

### 3.4 Create Training and Testing Sequences

In [ ]:
# Use 60 days of historical data to predict the next day
seq_length = 60

# Create sequences for training and testing
X_train, y_train = create_sequences(train_data, seq_length)
X_test, y_test = create_sequences(test_data, seq_length)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

### 3.5 Build LSTM Model

Create a multi-layer LSTM model with dropout layers to prevent overfitting.

In [ ]:
# Define LSTM model class in PyTorch
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=dropout)
        self.fc1 = nn.Linear(hidden_size, 25)
        self.fc2 = nn.Linear(25, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Initialize hidden state and cell state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        # Forward propagate LSTM
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]  # Get last time step
        
        # Pass through fully connected layers
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out

# Initialize model
model = LSTMModel()
print(model)

### 3.6 Train the LSTM Model

Train the model with early stopping to prevent overfitting.

In [ ]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

# Create data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Training batches: {len(train_loader)}")

### 3.7 Make Predictions and Evaluate LSTM

In [ ]:
# Train the model
num_epochs = 100
train_losses = []
best_loss = float('inf')
patience = 10
patience_counter = 0

print("Training LSTM model...")
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Early stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}')
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(torch.load('best_model.pth'))
print("Training complete!")

## 4. Method 2: ARIMA (Statistical Approach)

ARIMA (AutoRegressive Integrated Moving Average) is a classical statistical method for time series forecasting.

In [ ]:
# Make predictions
model.eval()
with torch.no_grad():
    train_predict = model(X_train_tensor).numpy()
    test_predict = model(X_test_tensor).numpy()

# Inverse transform predictions
train_predict = scaler.inverse_transform(train_predict)
test_predict = scaler.inverse_transform(test_predict)
y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculate evaluation metrics
train_rmse = np.sqrt(mean_squared_error(y_train_actual, train_predict))
test_rmse = np.sqrt(mean_squared_error(y_test_actual, test_predict))
test_mae = mean_absolute_error(y_test_actual, test_predict)

print("="*50)
print("LSTM Model Performance")
print("="*50)
print(f"Train RMSE: ${train_rmse:.2f}")
print(f"Test RMSE: ${test_rmse:.2f}")
print(f"Test MAE: ${test_mae:.2f}")
print("="*50)

### 4.1 Prepare Data for ARIMA

In [ ]:
# Use the same train/test split
train_arima = df['close'][:train_size]
test_arima = df['close'][train_size:]

print(f"ARIMA Training samples: {len(train_arima)}")
print(f"ARIMA Testing samples: {len(test_arima)}")

### 4.2 Fit ARIMA Model and Make Predictions

Using ARIMA(5,1,0) parameters:
- p=5: Number of lag observations
- d=1: Degree of differencing
- q=0: Size of moving average window

In [ ]:
# Fit ARIMA model with order (p=5, d=1, q=0)
model_arima = ARIMA(train_arima, order=(5, 1, 0))
fitted_arima = model_arima.fit()

# Display model summary
print(fitted_arima.summary())

### 4.3 Evaluate ARIMA Model

In [ ]:
# Forecast on test data
forecast_arima = fitted_arima.forecast(steps=len(test_arima))

# Calculate evaluation metrics
arima_rmse = np.sqrt(mean_squared_error(test_arima, forecast_arima))
arima_mae = mean_absolute_error(test_arima, forecast_arima)

print("="*50)
print("ARIMA Model Performance")
print("="*50)
print(f"Test RMSE: ${arima_rmse:.2f}")
print(f"Test MAE: ${arima_mae:.2f}")
print("="*50)

## 5. Method 3: Prophet (Facebook's Time Series Tool)

Prophet is a forecasting procedure designed to handle time series with strong seasonal effects and multiple seasons of historical data.

In [ ]:
from prophet import Prophet

### 5.1 Prepare Data for Prophet

Prophet requires data in a specific format with columns named 'ds' (date) and 'y' (value).

In [ ]:
# Prepare data in Prophet format
df_prophet = df.reset_index()[['date', 'close']]
df_prophet.columns = ['ds', 'y']

# Split into train and test sets
train_prophet = df_prophet[:train_size]
test_prophet = df_prophet[train_size:]

print(f"Prophet Training samples: {len(train_prophet)}")
print(f"Prophet Testing samples: {len(test_prophet)}")
train_prophet.head()

### 5.2 Fit Prophet Model and Make Predictions

In [ ]:
# Initialize and fit Prophet model
prophet_model = Prophet(daily_seasonality=True, yearly_seasonality=True)
prophet_model.fit(train_prophet)

# Create future dataframe and make predictions
future = prophet_model.make_future_dataframe(periods=len(test_prophet))
forecast_prophet = prophet_model.predict(future)

# Display forecast columns
forecast_prophet[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

### 5.3 Evaluate Prophet Model

In [ ]:
# Extract predictions for test period
prophet_pred = forecast_prophet['yhat'][-len(test_prophet):].values

# Calculate evaluation metrics
prophet_rmse = np.sqrt(mean_squared_error(test_prophet['y'], prophet_pred))
prophet_mae = mean_absolute_error(test_prophet['y'], prophet_pred)

print("="*50)
print("Prophet Model Performance")
print("="*50)
print(f"Test RMSE: ${prophet_rmse:.2f}")
print(f"Test MAE: ${prophet_mae:.2f}")
print("="*50)

## 6. Compare All Models

Compare the performance of all three models side by side.

In [ ]:
# Create comparison dataframe
comparison = pd.DataFrame({
    'Model': ['LSTM', 'ARIMA', 'Prophet'],
    'Test RMSE': [test_rmse, arima_rmse, prophet_rmse],
    'Test MAE': [test_mae, arima_mae, prophet_mae]
})

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison.to_string(index=False))
print("="*60)

# Identify best model
best_model = comparison.loc[comparison['Test RMSE'].idxmin(), 'Model']
print(f"\nBest Model (Lowest RMSE): {best_model}")

## 7. Visualize Predictions

Create comprehensive visualizations comparing actual vs predicted values for all three models.

In [ ]:
plt.figure(figsize=(15, 10))

# Plot 1: LSTM predictions
plt.subplot(3, 1, 1)
plt.plot(df.index[train_size+seq_length:], y_test_actual, label='Actual', linewidth=2, color='blue')
plt.plot(df.index[train_size+seq_length:], test_predict, label='LSTM Prediction', linewidth=2, color='red')
plt.title('LSTM Time Series Forecasting', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Stock Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: ARIMA predictions
plt.subplot(3, 1, 2)
plt.plot(test_arima.index, test_arima.values, label='Actual', linewidth=2, color='blue')
plt.plot(test_arima.index, forecast_arima.values, label='ARIMA Prediction', linewidth=2, color='green')
plt.title('ARIMA Time Series Forecasting', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Stock Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 3: Prophet predictions
plt.subplot(3, 1, 3)
plt.plot(test_prophet['ds'], test_prophet['y'], label='Actual', linewidth=2, color='blue')
plt.plot(test_prophet['ds'], prophet_pred, label='Prophet Prediction', linewidth=2, color='orange')
plt.title('Prophet Time Series Forecasting', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Stock Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('forecasting_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved as 'forecasting_results.png'")

## 8. Future Predictions (30 Days)

Use the LSTM model to predict stock prices for the next 30 days.

### 8.1 Generate 30-Day Forecast

In [ ]:
# Use the last sequence from the dataset
last_sequence = scaled_data[-seq_length:]
future_predictions = []

# Generate predictions for 30 days
model.eval()
with torch.no_grad():
    for _ in range(30):
        # Prepare input
        input_seq = torch.FloatTensor(last_sequence).unsqueeze(0)
        
        # Predict the next value
        pred = model(input_seq).item()
        future_predictions.append(pred)
        
        # Update sequence with the new prediction
        last_sequence = np.append(last_sequence[1:], [[pred]], axis=0)

# Inverse transform predictions to original scale
future_predictions = scaler.inverse_transform(np.array(future_predictions).reshape(-1, 1))

# Create future dates
future_dates = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1), periods=30, freq='D')

print("30-Day forecast generated successfully!")

### 8.2 Visualize 30-Day Forecast

In [ ]:
plt.figure(figsize=(14, 6))

# Plot historical data (last 100 days)
plt.plot(df.index[-100:], df['close'][-100:], label='Historical', linewidth=2, color='blue')

# Plot future predictions
plt.plot(future_dates, future_predictions, label='30-Day Forecast', linewidth=2.5, 
         linestyle='--', color='red', marker='o', markersize=4)

plt.title('Google Stock Price - 30 Day Forecast', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Stock Price ($)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('future_forecast.png', dpi=300, bbox_inches='tight')
plt.show()

print("Future forecast visualization saved as 'future_forecast.png'")

### 8.3 Display 30-Day Forecast Values

In [ ]:
# Create forecast dataframe
forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted Price ($)': future_predictions.flatten()
})

forecast_df['Date'] = forecast_df['Date'].dt.strftime('%Y-%m-%d')

print("\n" + "="*50)
print("30-DAY STOCK PRICE FORECAST")
print("="*50)
print(forecast_df.to_string(index=False))
print("="*50)

# Summary statistics
print(f"\nForecast Summary:")
print(f"  Starting Price: ${future_predictions[0][0]:.2f}")
print(f"  Ending Price: ${future_predictions[-1][0]:.2f}")
print(f"  Average Price: ${future_predictions.mean():.2f}")
print(f"  Price Change: ${future_predictions[-1][0] - future_predictions[0][0]:.2f} ({((future_predictions[-1][0] - future_predictions[0][0])/future_predictions[0][0]*100):.2f}%)")

## 9. Conclusion

This notebook demonstrated three different approaches to time series forecasting:

- **LSTM**: Best for capturing complex patterns and non-linear relationships
- **ARIMA**: Traditional statistical approach, good for linear trends
- **Prophet**: Handles seasonality well and is robust to missing data

The performance comparison shows which model works best for this particular dataset. The 30-day forecast provides insights into potential future stock price movements.

**Note**: Stock price predictions are for educational purposes only and should not be used for actual trading decisions.